# DR-DT Benchmarks

In [ ]:
!git clone https://github.com/Jakefawkes/DR_distributional_test code/

import sys
sys.path.insert(0, "code/DR_distributional_test")

In [ ]:
import gpytorch.kernels as kernels
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression

from src.data import Data_object
from src.test import kernel_permutation_test
from src.utils import compute_median_heuristic

from causal_medmnist import Scenario
from causal_medmnist.datasets import REGISTRY

In [ ]:
n_samples = 1000
num_repetitions = 100
n_permutations = 200
n_bins = 100
cme_reg = 0.001


def generate(scenario, n, rng, split):
    sample = scenario.generate(n, seed=int(rng.integers(0, 2**32)), split=split, replace=True)
    X = torch.tensor(sample.X, dtype=torch.float32)
    Y = torch.tensor(sample.Y.reshape(n, -1), dtype=torch.float32)
    T = torch.tensor(sample.A, dtype=torch.float32)
    return Data_object(X, Y, T)


def run_test(dataset, n_samples, scale, rng):
    scenario = Scenario(dataset, effect_strength=scale)
    rejections = []

    for _ in range(num_repetitions):
        data_train = generate(scenario, n_samples // 2, rng, "train")
        data_test = generate(scenario, n_samples - n_samples // 2, rng, "val")
        data = data_train.join(data_test)

        X_ker = kernels.RBFKernel(ard_num_dims=data.X.shape[1])
        Y_ker = kernels.RBFKernel(ard_num_dims=data.Y.shape[1])
        X_ker.lengthscale = compute_median_heuristic(data.X)
        Y_ker.lengthscale = compute_median_heuristic(data.Y)

        result = kernel_permutation_test(
            data_train, data_test, X_ker, Y_ker, LogisticRegression(),
            test_stat="DATE", n_bins=n_bins, n_permutations=n_permutations, permute_weights=True, reg=[cme_reg, cme_reg],
        )
        rejections.append(float(result["p_val"] < 0.05))

    return np.mean(rejections)


def run(scale, rng):
    for dataset in sorted(REGISTRY):
        result = run_test(dataset, n_samples, scale, rng)
        print(f"[{dataset}][N={n_samples}][scale={scale}] {result}")

In [ ]:
rng = np.random.default_rng(0)

run(scale=0.6, rng=rng)
run(scale=0.0, rng=rng)